In [1]:
# =========================================================
# STEP 0: INSTALL & ENV SETUP
# =========================================================
print("\n[STEP 0] Setting up environment...")

!pip install -q timm grad-cam

import warnings
warnings.filterwarnings("ignore")

print("[STEP 0] Environment ready ✅")


[STEP 0] Setting up environment...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 47.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[STEP 0] Environment ready ✅


In [ ]:
# =========================================================
# STEP 1: MOUNT GOOGLE DRIVE
# =========================================================
print("\n[STEP 1] Mounting Google Drive...")

from google.colab import drive
drive.mount('/content/drive')

print("[STEP 1] Google Drive mounted ✅")

In [ ]:
# =========================================================
# STEP 2: IMPORT LIBRARIES
# =========================================================
print("\n[STEP 2] Importing libraries...")

import os, random, re
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset

import timm
from torchvision import transforms
from torchvision.transforms import RandomErasing

from sklearn.model_selection import GroupShuffleSplit

print("[STEP 2] Libraries imported ✅")

In [ ]:
# =========================================================
# STEP 3: CONFIGURATION
# =========================================================
print("\n[STEP 3] Loading configuration...")

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5   # This is the default, but will be iterated over by EPOCHS_LIST
LR = 2e-4
SEEDS = [42, 123] # Original seeds
NEW_SEEDS = [24, 88]

# Extend SEEDS to include NEW_SEEDS for full evaluation
ALL_SEEDS = SEEDS # Changed to use only initial SEEDS

EPOCHS_LIST = [5, 10] # For experimentation with different epoch counts
UNFREEZE_EPOCH = 3 # Epoch to unfreeze backbone
USE_FOCAL = True # Whether to use Focal Loss
USE_SAMPLER = True # Whether to use WeightedRandomSampler for class imbalance
USE_ERASE = True # Whether to use RandomErasing in transforms

DATA_ROOT = "/content/drive/MyDrive/Blood_Cancer_Detection_Project_Mentor/Blood_cell_Cancer_[ALL]_Dataset"
RESULTS_DIR = "/content/drive/MyDrive/ALL_MULTI_MODEL_RESEARCH"
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_FOLDERS = {
    "Benign": "Benign",
    "Early-PreB": "[Malignant] early Pre-B",
    "Pre-B": "[Malignant] Pre-B",
    "Pro-B": "[Malignant] Pro-B"
}

NUM_CLASSES = len(CLASS_FOLDERS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ZOO = {
    "MobileNetV3": "mobilenetv3_large_100",
    "EfficientNetB0": "efficientnet_b0",
    "ResNet50": "resnet50",
    "ViT_Tiny": "vit_tiny_patch16_224" # Adding a Vision Transformer
}

print(f"[STEP 3] Config loaded | Device: {DEVICE} ✅")


In [ ]:
# =========================================================
# STEP 4: REPRODUCIBILITY
# =========================================================
print("\n[STEP 4] Setting seeds...")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("[STEP 4] Seed function ready ✅")

In [ ]:
# =========================================================
# STEP 5: DATA LOADING
# =========================================================
print("\n[STEP 5] Loading dataset...")

def gather_images(folder):
    return sorted([
        str(p) for p in Path(folder).rglob("*")
        if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
    ])

def infer_group(path):
    m = re.search(r'(\d{3,})', os.path.basename(path))
    return m.group(1) if m else Path(path).parent.name

paths, labels, groups = [], [], []

for idx, cls in enumerate(CLASS_FOLDERS):
    folder = os.path.join(DATA_ROOT, CLASS_FOLDERS[cls])
    for p in gather_images(folder):
        try:
            Image.open(p).convert("RGB")
            paths.append(p)
            labels.append(idx)
            groups.append(infer_group(p))
        except:
            continue

print(f"[STEP 5] Total images: {len(paths)} ✅")

# Dataset distribution
print("\n[DATASET DISTRIBUTION]")
counts = Counter(labels)
for i, name in enumerate(CLASS_FOLDERS.keys()):
    print(f"{name}: {counts[i]}")

In [ ]:
# =========================================================
# STEP 6: PATIENT-WISE SPLIT (FIXED)
# =========================================================
print("\n[STEP 6] Creating patient-wise splits...")

def train_val_test_split(paths, labels, groups, seed):

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(paths, labels, groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=seed)
    train_idx_rel, val_idx_rel = next(gss2.split(
        [paths[i] for i in trainval_idx],
        [labels[i] for i in trainval_idx],
        [groups[i] for i in trainval_idx]
    ))

    # ✅ FIX: Map back to original indices
    train_idx = [trainval_idx[i] for i in train_idx_rel]
    val_idx = [trainval_idx[i] for i in val_idx_rel]

    def select(arr, idx):
        return [arr[i] for i in idx]

    return (
        select(paths, train_idx),
        select(labels, train_idx),
        select(paths, val_idx),
        select(labels, val_idx),
        select(paths, test_idx),
        select(labels, test_idx)
    )

# Test split
train_files, train_labels, val_files, val_labels, test_files, test_labels = \
    train_val_test_split(paths, labels, groups, seed=42)

print("[STEP 6] Split created ✅")

print(f"Train: {len(train_labels)}")
print(f"Val: {len(val_labels)}")
print(f"Test: {len(test_labels)}")

In [ ]:
# =========================================================
# STEP 7: TRANSFORMS (FIXED)
# =========================================================
print("\n[STEP 7] Creating transforms...")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1,0.1,0.1),
    transforms.ToTensor(),
    RandomErasing(p=0.25),   # ✅ FIXED (now used)
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

print("[STEP 7] Transforms ready ✅")

In [ ]:
# =========================================================
# STEP 7: DATASET CLASS (FIXED)
# =========================================================
print("\n[STEP 7] Creating Dataset class...")

class ALLDataset(Dataset):
    def __init__(self, files, labels, tf):
        self.files = files
        self.labels = labels
        self.tf = tf

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        img = self.tf(img)
        return img, self.labels[idx]

print("[STEP 7] Dataset class ready ✅")

In [ ]:
# =========================================================
# STEP 8: LOSS FUNCTION
# =========================================================
print("\n[STEP 8] Setting loss function...")

class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

criterion = FocalLoss() if USE_FOCAL else nn.CrossEntropyLoss()

print("[STEP 8] Loss function ready ✅")

In [ ]:
# =========================================================
# STEP 9: EVALUATION FUNCTION
# =========================================================
print("\n[STEP 9] Preparing evaluation function...")

def evaluate_model(model, loader):
    model.eval()
    preds, probs, gts = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            prob = torch.softmax(out, 1)

            preds.extend(out.argmax(1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            gts.extend(yb.cpu().numpy())

    preds = np.array(preds)
    gts = np.array(gts)
    probs = np.array(probs)

    acc = accuracy_score(gts, preds)
    bal_acc = balanced_accuracy_score(gts, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        gts, preds, average="macro"
    )

    auc = roc_auc_score(
        label_binarize(gts, classes=range(NUM_CLASSES)),
        probs, average="macro", multi_class="ovr"
    )

    return {
        "acc": acc,
        "bal_acc": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "kappa": cohen_kappa_score(gts, preds),
        "mcc": matthews_corrcoef(gts, preds),
        "cm": confusion_matrix(gts, preds),
        "gts": gts.tolist(),
        "probs": probs.tolist()
    }

print("[STEP 9] Evaluation function ready ✅")

In [ ]:
# =========================================================
# STEP 10: TRAINING + EVALUATION LOOP (FIXED)
# =========================================================
print("\n[STEP 10] Starting experiments...")

from collections import defaultdict
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, cohen_kappa_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import json # Import json for saving history

final_results = defaultdict(list)

# \u25ba Optional (for reviewer justification)
EPOCHS_LIST = [5]   # You can later change to [5, 10]

for SEED in ALL_SEEDS: # Use ALL_SEEDS to include new seeds in the initial training
    print(f"\n===== SEED {SEED} ====")

    # Determine epochs for the current seed
    if ALL_SEEDS.index(SEED) == 0: # First seed
        current_epochs = 10
    else: # Second seed
        current_epochs = 5
    print(f"Using {current_epochs} epochs for this seed.")

    # \u2705 FIXED: Proper reproducibility
    set_seed(SEED)

    # Use previously defined CORRECT split function
    train_files, train_labels, val_files, val_labels, test_files, test_labels = \
        train_val_test_split(paths, labels, groups, SEED)

    # =========================
    # SAMPLER (CLASS IMBALANCE)
    # =========================
    sampler = None
    if USE_SAMPLER:
        counts = Counter(train_labels)
        weights = torch.DoubleTensor([1.0 / counts[l] for l in train_labels])
        sampler = WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True
        )

    # =========================
    # DATALOADERS
    # =========================
    train_loader = DataLoader(
        ALLDataset(train_files, train_labels, train_tf),
        batch_size=BATCH_SIZE,
        sampler=sampler,
        shuffle=(sampler is None),
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        ALLDataset(val_files, val_labels, val_tf),
        batch_size=BATCH_SIZE
    )

    test_loader = DataLoader(
        ALLDataset(test_files, test_labels, val_tf),
        batch_size=BATCH_SIZE
    )

    # =========================
    # MODEL LOOP
    # =========================
    for model_name, timm_name in MODEL_ZOO.items():
        print(f"\n--- Training {model_name} ---")

        model = timm.create_model(
            timm_name, pretrained=True, num_classes=NUM_CLASSES
        ).to(DEVICE)

        # Freeze backbone
        for n, p in model.named_parameters():
            if not any(k in n for k in ("classifier", "fc", "head")):
                p.requires_grad = False

        optimizer = AdamW(
            filter(lambda p: p.requires_grad, model.parameters()), lr=LR
        )
        scaler = GradScaler()

        best_auc = 0
        best_path = f"{RESULTS_DIR}/{model_name}_SEED{SEED}.pth"
        history_path = f"{RESULTS_DIR}/{model_name}_SEED{SEED}_history.json" # Path for history file

        history = {
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "val_auc": []
        }

        # =========================
        # TRAINING LOOP
        # =========================
        for epoch in range(1, current_epochs + 1):

            print(f"\n[Epoch {epoch}/{current_epochs}]")

            model.train()
            train_loss, train_correct, train_total = 0, 0, 0

            if epoch == UNFREEZE_EPOCH:
                print("\ud83d\udd13 Unfreezing backbone")
                for p in model.parameters():
                    p.requires_grad = True
                optimizer = AdamW(model.parameters(), lr=LR * 0.2)

            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()

                with autocast(enabled=(DEVICE.type == "cuda")):
                    out = model(xb)
                    loss = criterion(out, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                train_loss += loss.item() * xb.size(0)
                train_correct += (out.argmax(1) == yb).sum().item()
                train_total += yb.size(0)

            train_loss /= train_total
            train_acc = train_correct / train_total

            # =========================
            # VALIDATION
            # =========================
            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0
            preds, probs, gts = [], [], []

            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)

                    out = model(xb)
                    loss = criterion(out, yb)
                    prob = torch.softmax(out, 1)

                    val_loss += loss.item() * xb.size(0)
                    val_correct += (out.argmax(1) == yb).sum().item()
                    val_total += yb.size(0)

                    preds.extend(out.argmax(1).cpu().numpy())
                    probs.extend(prob.cpu().numpy())
                    gts.extend(yb.cpu().numpy())

            val_loss /= val_total
            val_acc = val_correct / val_total

            try:
                val_auc = roc_auc_score(
                    label_binarize(gts, classes=range(NUM_CLASSES)),
                    np.array(probs),
                    average="macro",
                    multi_class="ovr"
                )
            except:
                val_auc = np.nan

            # Save history
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["val_auc"].append(val_auc) # Add AUC to history

            print(f"Train Loss: {train_loss:.4f}")
            print(f"Val Loss  : {val_loss:.4f}")
            print(f"Val AUC   : {val_auc:.4f}")

            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), best_path)
                # Save history along with the best model
                with open(history_path, 'w') as f:
                    json.dump(history, f, indent=4)
                print("\ud83d\udcbe Best model and history saved")

        # =========================
        # TEST EVALUATION
        # =========================
        # Load the best model weights if they were saved
        if os.path.exists(best_path):
            model.load_state_dict(torch.load(best_path))
        else:
            print(f"Warning: No best model checkpoint found at {best_path}. Using last epoch's model for test evaluation.")

        test_results = evaluate_model(model, test_loader)
        final_results[model_name].append(test_results)

        print(f"\n[Test AUC]: {test_results['auc']:.4f}")

In [ ]:
# =========================================================
# STEP 11: GRAD-CAM VISUALIZATION (FIXED & REVIEWER-READY)
# =========================================================
print("\n[STEP 11] Generating Grad-CAM Visualizations...")

from datetime import datetime
# Add missing imports for Grad-CAM utilities
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import timm
from PIL import Image
from torchvision import transforms

# Re-define necessary configs if this cell might be run independently for review
# (though it's better practice to ensure they are loaded from prior cells)
# Assuming IMG_SIZE, CLASS_FOLDERS, NUM_CLASSES, DEVICE, RESULTS_DIR, MODEL_ZOO, val_tf
# and train_val_test_split function are defined in preceding executed cells.

# Define a function to get the appropriate target layer for Grad-CAM for different models
def get_target_layer(model_name, model):
    if "MobileNetV3" in model_name:
        return model.conv_head # Last convolutional block before the classifier
    elif "EfficientNetB0" in model_name:
        return model.conv_head # Common for timm EfficientNets
    elif "ResNet50" in model_name:
        return model.layer4 # The last block of residual layers in ResNet
    elif "ViT_Tiny" in model_name:
        # For Vision Transformers, target the last normalization layer before the classifier
        return model.norm
    else:
        # Fallback for unknown models - try common layers
        print(f"Warning: No specific target layer defined for {model_name}. Attempting to use model.norm or last block.")
        if hasattr(model, 'norm'):
            return model.norm
        elif hasattr(model, 'blocks') and len(model.blocks) > 0: # For models with a block structure
            return model.blocks[-1]
        else:
            raise ValueError(f"Could not find a suitable target layer for {model_name}. Please specify manually.")

def generate_gradcam_batch(model_instance, current_model_name, files, labels, save_root):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Make a model-specific directory for Grad-CAM outputs
    model_save_dir = os.path.join(save_root, f"gradcam_{current_model_name}_{timestamp}")

    correct_dir = os.path.join(model_save_dir, "correct")
    wrong_dir = os.path.join(model_save_dir, "misclassified")

    os.makedirs(correct_dir, exist_ok=True)
    os.makedirs(wrong_dir, exist_ok=True)

    correct_count, wrong_count = 0, 0

    print(f"Saving Grad-CAM outputs for {current_model_name} to: {model_save_dir}")

    class_names = list(CLASS_FOLDERS.keys())

    # Get the target layer for the current model
    try:
        target_layer = get_target_layer(current_model_name, model_instance)
    except ValueError as e:
        print(f"Skipping Grad-CAM for {current_model_name} due to: {e}")
        return

    # Using GradCAM as the CAM method
    cam_algorithm = GradCAM

    for i in range(len(files)): # Iterate through test samples
        img_path = files[i]
        true_label = labels[i]

        img = Image.open(img_path).convert("RGB")
        input_tensor = val_tf(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model_instance(input_tensor)
            pred_class = output.argmax().item()

        targets = [ClassifierOutputTarget(pred_class)]

        # Ensure target_layers is a list for GradCAM
        with cam_algorithm(model=model_instance, target_layers=[target_layer]) as cam:
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        img_np = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
        cam_img = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        gt_name = class_names[true_label]
        pred_name = class_names[pred_class]

        # Save logic for up to 5 correct and 5 wrong predictions
        if pred_class == true_label and correct_count < 5:
            filename = f"CORRECT_{correct_count+1}_GT-{gt_name}_PRED-{pred_name}.png"
            save_path = os.path.join(correct_dir, filename)
            correct_count += 1

        elif pred_class != true_label and wrong_count < 5:
            filename = f"WRONG_{wrong_count+1}_GT-{gt_name}_PRED-{pred_name}.png"
            save_path = os.path.join(wrong_dir, filename)
            wrong_count += 1
        else:
            # If we've already saved enough for this category or it's neither
            if correct_count >= 5 and wrong_count >= 5:
                break
            continue # Continue to next image if not saved

        plt.figure(figsize=(4,4))
        plt.imshow(cam_img)
        plt.title(f"GT: {gt_name} | Pred: {pred_name}")
        plt.axis("off")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()

    print(f"✅ Saved {correct_count} correct & {wrong_count} misclassified samples for {current_model_name}")


# Regenerate split (avoid dependency on loop state)
# Assuming 'paths', 'labels', 'groups' are loaded from previous cells
train_files, train_labels, val_files, val_labels, test_files, test_labels = \
    train_val_test_split(paths, labels, groups, seed=42) # Using seed 42 for consistency with model loading

# Loop through all models in MODEL_ZOO to generate Grad-CAMs
for model_name, timm_name in MODEL_ZOO.items():
    print(f"\n--- Generating Grad-CAM for {model_name} ---")

    # Load the best model (using SEED 42's checkpoint for consistency)
    model_instance = timm.create_model(
        timm_name, pretrained=False, num_classes=NUM_CLASSES
    )
    # Check if a model checkpoint exists for this model and seed (e.g., from original SEED 42 run)
    model_checkpoint_path = f"{RESULTS_DIR}/{model_name}_SEED42.pth"
    if os.path.exists(model_checkpoint_path):
        model_instance.load_state_dict(torch.load(model_checkpoint_path, map_location=DEVICE))
        model_instance.to(DEVICE)
        model_instance.eval()
        generate_gradcam_batch(model_instance, model_name, test_files, test_labels, RESULTS_DIR)
    else:
        print(f"Skipping Grad-CAM for {model_name}: No checkpoint found at {model_checkpoint_path}")

print("\n[STEP 11] Grad-CAM generation for all models completed ✅")

In [ ]:
# =========================================================
# STEP 12: FINAL TEST RESULTS (MEAN ± STD)
# =========================================================
print("\n=========== FINAL TEST RESULTS (MEAN ± STD) ==========")

for model, runs in final_results.items():
    print(f"\nModel: {model}")
    for metric in ["acc", "bal_acc", "precision", "recall", "f1", "auc", "kappa", "mcc"]:
        vals = [r[metric] for r in runs]
        print(f"{metric.upper():10s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

In [ ]:
# =========================================================
# STEP 13: AGGREGATED RESULTS
# =========================================================
print("\n[STEP 13] Aggregating results...")

aggregated_results = {}

for model_name, runs in final_results.items():
    aggregated_results[model_name] = {}
    for metric in ["acc", "bal_acc", "precision", "recall", "f1", "auc", "kappa", "mcc"]:
        vals = [r[metric] for r in runs]
        aggregated_results[model_name][metric] = {
            "mean": np.mean(vals),
            "std": np.std(vals)
        }

for model_name, metrics in aggregated_results.items():
    print(f"\nModel: {model_name}")
    for metric, stats in metrics.items():
        print(f"{metric.upper():10s}: {stats['mean']:.4f} ± {stats['std']:.4f}")

print("[STEP 13] Aggregation complete ✅")

In [ ]:
# =========================================================
# STEP 14: RESULTS DATAFRAME
# =========================================================
print("\n[STEP 14] Creating results DataFrame...")

import pandas as pd

df_results = []

for model_name, metrics in aggregated_results.items():
    row = {"Model": model_name}
    for metric, stats in metrics.items():
        row[f"{metric.upper()} Mean"] = stats["mean"]
        row[f"{metric.upper()} Std"] = stats["std"]
    df_results.append(row)

df_results = pd.DataFrame(df_results)

print("\nFinal Results Table:")
print(df_results)

print("[STEP 14] DataFrame ready ✅")

In [ ]:
# =========================================================
# STEP 15: PERFORMANCE METRIC BAR PLOTS
# =========================================================
print("\n[STEP 15] Generating performance metric bar plots...")

import matplotlib.pyplot as plt
import seaborn as sns

metrics_to_plot = ["AUC", "BAL_ACC", "F1"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(15, 5))

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    means = df_results[f"{metric} Mean"].values
    stds = df_results[f"{metric} Std"].values
    models = df_results["Model"].values

    ax.bar(models, means, yerr=stds, capsize=5, color=sns.color_palette("viridis", len(models)))
    ax.set_title(f"Model Comparison: {metric} (Mean ± Std)")
    ax.set_ylabel(f"{metric} Mean")
    ax.set_xlabel("Model")
    ax.set_ylim(0.8, 1.0)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("[STEP 15] Performance metric bar plots generated ✅")

In [ ]:
# =========================================================
# STEP 16: STORE INITIAL RESULTS TO JSON
# =========================================================
print("\n[STEP 16] Storing initial results to JSON...")

import json
from datetime import datetime
import numpy as np

def convert_numpy_to_list(obj):
    """Recursively converts numpy arrays and types to Python lists/scalars."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_list(e) for e in obj]
    elif isinstance(obj, (np.int64, np.float64, np.bool_)):
        return obj.item()
    return obj

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

serializable_final_results = {
    model_name: [convert_numpy_to_list(run_data) for run_data in runs]
    for model_name, runs in final_results.items()
}

serializable_aggregated_results = convert_numpy_to_list(aggregated_results)

all_results_data = {
    "timestamp": timestamp,
    "final_results": serializable_final_results,
    "aggregated_results": serializable_aggregated_results,
    "config": {
        "IMG_SIZE": IMG_SIZE,
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "LR": LR,
        "UNFREEZE_EPOCH": UNFREEZE_EPOCH,
        "SEEDS": SEEDS,
        "USE_FOCAL": USE_FOCAL,
        "USE_ERASE": USE_ERASE,
        "USE_SAMPLER": USE_SAMPLER,
        "DATA_ROOT": DATA_ROOT,
        "RESULTS_DIR": RESULTS_DIR,
        "CLASS_FOLDERS": CLASS_FOLDERS,
        "MODEL_ZOO": MODEL_ZOO,
        "NUM_CLASSES": NUM_CLASSES
    }
}

output_filename = f"{RESULTS_DIR}/initial_results_{timestamp}.json"

with open(output_filename, 'w') as f:
    json.dump(all_results_data, f, indent=4)

print(f"[STEP 16] Initial results stored to: {output_filename} ✅")

In [ ]:
# =========================================================
# STEP 17: EVALUATE BEST PERFORMING MODEL WITH NEW SEEDS
# =========================================================
print(f"\n[STEP 17] Evaluating {best_model_name_overall} with new seeds...")

# Use the NEW_SEEDS defined in configuration (STEP 3)
# NEW_SEEDS = [24, 88] is already defined in STEP 3, no need to redefine here

if best_model_name_overall is None:
    print("[STEP 17] No best model identified from initial runs. Skipping re-evaluation.")
else:
    for SEED in NEW_SEEDS: # This will iterate over [24, 88]
        print(f"\n===== SEED {SEED} ===telek")
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)

        train_files, train_labels, val_files, val_labels, test_files, test_labels = \
            train_val_test_split(paths, labels, groups, SEED)

        sampler = None
        if USE_SAMPLER:
            counts = Counter(train_labels)
            weights = torch.DoubleTensor([1.0 / counts[l] for l in train_labels])
            sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

        train_loader = DataLoader(
            ALLDataset(train_files, train_labels, train_tf), # Removed USE_ERASE as it's not in ALLDataset's __init__
            batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True
        )

        val_loader = DataLoader(
            ALLDataset(val_files, val_labels, val_tf),
            batch_size=BATCH_SIZE
        )

        test_loader = DataLoader(
            ALLDataset(test_files, test_labels, val_tf),
            batch_size=BATCH_SIZE
        )

        # --- Train the best model ---
        model_name_to_evaluate = best_model_name_overall # Use the identified best model
        timm_name = MODEL_ZOO[model_name_to_evaluate]

        model = timm.create_model(timm_name, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

        for n, p in model.named_parameters():
            if not any(k in n for k in ("classifier", "fc", "head")):
                p.requires_grad = False

        optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
        scaler = GradScaler()
        best_auc = 0
        current_best_path = f"{RESULTS_DIR}/{model_name_to_evaluate}_SEED{SEED}_new.pth" # Use the identified best model name

        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_loss, train_correct, train_total = 0, 0, 0

            if epoch == UNFREEZE_EPOCH:
                print("🔓 Unfreezing backbone")
                for p in model.parameters():
                    p.requires_grad = True
                optimizer = AdamW(model.parameters(), lr=LR * 0.2)

            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                with autocast(enabled=(DEVICE.type=="cuda")):
                    out = model(xb)
                    loss = criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                train_loss += loss.item() * xb.size(0)
                train_correct += (out.argmax(1) == yb).sum().item()
                train_total += yb.size(0)

            train_loss /= train_total
            train_acc = train_correct / train_total

            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0
            preds, probs, gts = [], [], []

            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    out = model(xb)
                    loss = criterion(out, yb)
                    prob = torch.softmax(out, 1)
                    val_loss += loss.item() * xb.size(0)
                    val_correct += (out.argmax(1) == yb).sum().item()
                    val_total += yb.size(0)
                    preds.extend(out.argmax(1).cpu().numpy())
                    probs.extend(prob.cpu().numpy())
                    gts.extend(yb.cpu().numpy())

            val_loss /= val_total
            val_acc = val_correct / val_total

            try:
                val_auc = roc_auc_score(label_binarize(gts, classes=range(NUM_CLASSES)),
                                        np.array(probs), average="macro", multi_class="ovr")
            except ValueError:
                val_auc = np.nan

            lr_current = optimizer.param_groups[0]["lr"]
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, LR: {lr_current:.2e}")

            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), current_best_path)
                print("💾 Best model updated ✔")
            else:
                print("No improvement ❌")

        # Evaluate best model on test set
        model.load_state_dict(torch.load(current_best_path))
        final_results[model_name_to_evaluate].append(evaluate_model(model, test_loader)) # Use the identified best model name

print("[STEP 17] Evaluation with new seeds complete ✅")


In [ ]:
# =========================================================
# STEP 18: RE-AGGREGATE RESULTS INCLUDING NEW SEEDS
# =========================================================
print("\n[STEP 18] Re-aggregating results with new seeds...")

aggregated_results = {}

for model_name, runs in final_results.items():
    aggregated_results[model_name] = {}
    for metric in ["acc", "bal_acc", "precision", "recall", "f1", "auc", "kappa", "mcc"]:
        vals = [run[metric] for run in runs]
        aggregated_results[model_name][metric] = {
            "mean": np.mean(vals),
            "std": np.std(vals)
        }

for model_name, metrics in aggregated_results.items():
    print(f"\nModel: {model_name}")
    for metric, stats in metrics.items():
        print(f"{metric.upper():10s}: {stats['mean']:.4f} ± {stats['std']:.4f}")

print("[STEP 18] Re-aggregation complete ✅")

In [ ]:
# =========================================================
# STEP 19: RE-CREATE DATAFRAME FROM RE-AGGREGATED RESULTS
# =========================================================
print("\n[STEP 19] Re-creating DataFrame from re-aggregated results...")

df_results_updated = []

for model_name, metrics in aggregated_results.items():
    row = {"Model": model_name}
    for metric, stats in metrics.items():
        row[f"{metric.upper()} Mean"] = stats["mean"]
        row[f"{metric.upper()} Std"] = stats["std"]
    df_results_updated.append(row)

df_results_updated = pd.DataFrame(df_results_updated)

print("\nDataFrame of Updated Aggregated Results:")
print(df_results_updated.head())

print("[STEP 19] DataFrame re-creation complete ✅")

In [ ]:
# =========================================================
# STEP 20: RE-GENERATE PERFORMANCE METRIC BAR PLOTS WITH UPDATED RESULTS
# =========================================================
print("\n[STEP 20] Re-generating performance metric bar plots with updated results...")

metrics_to_plot = ["AUC", "BAL_ACC", "F1"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(15, 5))

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    means = df_results_updated[f"{metric} Mean"].values
    stds = df_results_updated[f"{metric} Std"].values
    models = df_results_updated["Model"].values

    ax.bar(models, means, yerr=stds, capsize=5, color=sns.color_palette("viridis", len(models)))
    ax.set_title(f"Model Comparison: {metric} (Mean ± Std)")
    ax.set_ylabel(f"{metric} Mean")
    ax.set_xlabel("Model")
    ax.set_ylim(0.8, 1.0)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("[STEP 20] Performance metric bar plots re-generated ✅")

In [ ]:
# =========================================================
# STEP 21: FINAL COMPREHENSIVE ANALYSIS + JSON PLOTTING
# =========================================================
print('\n[STEP 21] Starting comprehensive analysis (inference + JSON plotting)...')

import os, re, glob, json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix,
    roc_curve, auc
)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# =========================================================
# 0. CONFIGURATION
# =========================================================
DATA_ROOT = "/content/drive/MyDrive/Blood_Cancer_Detection_Project_Mentor/Blood_cell_Cancer_[ALL]_Dataset"
RESULTS_DIR = "/content/drive/MyDrive/ALL_MULTI_MODEL_RESEARCH"
os.makedirs(RESULTS_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_FOLDERS = {
    "Benign": "Benign",
    "Early-PreB": "[Malignant] early Pre-B",
    "Pre-B": "[Malignant] Pre-B",
    "Pro-B": "[Malignant] Pro-B"
}

MODEL_ZOO = {
    "MobileNetV3": "mobilenetv3_large_100",
    "EfficientNetB0": "efficientnet_b0",
    "ResNet50": "resnet50",
    "ViT_Tiny": "vit_tiny_patch16_224" # Ensure all models are included
}

ORIGINAL_SEEDS = [42, 123]
NUM_CLASSES = len(CLASS_FOLDERS)
CLASS_NAMES = list(CLASS_FOLDERS.keys())

print(f"[STEP 21] Configured. Device: {DEVICE}, Num Classes: {NUM_CLASSES} ✅")

# =========================================================
# 1. TRANSFORMS AND DATASET CLASS
# =========================================================
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class ALLDataset(Dataset):
    def __init__(self, files, labels, tf):
        self.files = files
        self.labels = labels
        self.tf = tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        return self.tf(img), self.labels[idx]

# =========================================================
# 2. DATA COLLECTION
# =========================================================
def gather_images(folder):
    return sorted([str(p) for p in Path(folder).rglob("*")
                   if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}])

def infer_group(path):
    m = re.search(r'(\d{3,})', os.path.basename(path))
    return m.group(1) if m else Path(path).parent.name

paths, labels, groups = [], [], []
for idx, cls in enumerate(CLASS_FOLDERS):
    folder = os.path.join(DATA_ROOT, CLASS_FOLDERS[cls])
    for p in gather_images(folder):
        try:
            Image.open(p).convert("RGB")
            paths.append(p)
            labels.append(idx)
            groups.append(infer_group(p))
        except:
            continue

print(f"[STEP 21] Dataset loaded: {len(paths)} images ✅")

# =========================================================
# 3. TRAIN/VAL/TEST SPLIT FUNCTION
# =========================================================
def train_val_test_split(paths, labels, groups, seed):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(paths, labels, groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=seed)
    train_idx, val_idx = next(gss2.split(
        [paths[i] for i in trainval_idx],
        [labels[i] for i in trainval_idx],
        [groups[i] for i in trainval_idx]
    ))
    def sel(arr, idx): return [arr[i] for i in idx]
    return (
        sel(paths, train_idx), sel(labels, train_idx),
        sel(paths, val_idx), sel(labels, val_idx),
        sel(paths, test_idx), sel(labels, test_idx)
    )

# =========================================================
# 4. MODEL EVALUATION FUNCTION
# =========================================================
def evaluate_model(model, loader):
    model.eval()
    preds, probs, gts = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            prob = torch.softmax(out, 1)
            preds.extend(out.argmax(1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            gts.extend(yb.cpu().numpy())
    preds = np.array(preds)
    gts = np.array(gts)
    probs = np.array(probs)
    acc = accuracy_score(gts, preds)
    bal_acc = balanced_accuracy_score(gts, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(gts, preds, average="macro")
    auc_score = roc_auc_score(label_binarize(gts, classes=range(NUM_CLASSES)), probs, average="macro", multi_class="ovr")
    return {
        "acc": acc, "bal_acc": bal_acc, "precision": precision, "recall": recall,
        "f1": f1, "auc": auc_score, "kappa": cohen_kappa_score(gts, preds),
        "mcc": matthews_corrcoef(gts, preds), "cm": confusion_matrix(gts, preds),
        "gts": gts.tolist(), "probs": probs.tolist()
    }

# =========================================================
# 5. COLLECT ALL MODEL RESULTS (CHECKPOINTS + JSON)
# =========================================================
all_models_results = defaultdict(list)

# 5a. Load from .pth models
model_files = glob.glob(os.path.join(RESULTS_DIR, '*.pth'))
for model_file in model_files:
    basename = os.path.basename(model_file)
    match = re.match(r'(MobileNetV3|EfficientNetB0|ResNet50|ViT_Tiny)_SEED(\d+)(_new)?\.pth', basename)
    if match:
        m_name, s_id = match.group(1), int(match.group(2))
        torch.manual_seed(s_id); np.random.seed(s_id); random.seed(s_id)
        _, _, _, _, test_files, test_labels = train_val_test_split(paths, labels, groups, s_id)
        test_loader = DataLoader(ALLDataset(test_files, test_labels, val_tf),
                                 batch_size=BATCH_SIZE, shuffle=False)
        model = timm.create_model(MODEL_ZOO[m_name], pretrained=False, num_classes=NUM_CLASSES).to(DEVICE)
        model.load_state_dict(torch.load(model_file, map_location=DEVICE))
        model.eval()
        all_models_results[m_name].append(evaluate_model(model, test_loader))

# 5b. Load from previously saved JSON (if exists)
list_of_files = glob.glob(os.path.join(RESULTS_DIR, 'initial_results_*.json'))
if list_of_files:
    latest_json = max(list_of_files, key=os.path.getctime)
    print(f"[STEP 21] Loading previous JSON results: {latest_json}")
    with open(latest_json, 'r') as f:
        loaded_data = json.load(f)
    final_results_loaded = loaded_data['final_results']
    for model_name, runs in final_results_loaded.items():
        all_models_results[model_name].extend(runs)

print("[STEP 21] All model inference results collected ✅")

# 5c. Load history from JSON files
all_histories = defaultdict(lambda: defaultdict(dict)) # {model_name: {seed: history_dict}}
history_files = glob.glob(os.path.join(RESULTS_DIR, '*_history.json'))
for history_file in history_files:
    basename = os.path.basename(history_file)
    match = re.match(r'(MobileNetV3|EfficientNetB0|ResNet50|ViT_Tiny)_SEED(\d+)(_new)?_history\.json', basename)
    if match:
        m_name, s_id = match.group(1), int(match.group(2))
        with open(history_file, 'r') as f:
            history_data = json.load(f)
        all_histories[m_name][s_id] = history_data
print("[STEP 21] All training histories collected ✅")

# =========================================================
# 6. NEW VISUALIZATIONS: EPOCH HISTORY & CONFUSION MATRICES
# =========================================================

def plot_epoch_history(all_histories, model_name, seed):
    if model_name not in all_histories or seed not in all_histories[model_name]:
        print(f"No history data for {model_name} (Seed {seed})")
        return

    history = all_histories[model_name][seed]
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(18, 5)) # Increased figure size for 3 plots

    # Plot Loss
    plt.subplot(1, 3, 1) # Changed to 1 row, 3 columns
    plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o', linestyle='-')
    plt.plot(epochs, history['val_loss'], label='Validation Loss', marker='o', linestyle='--')
    plt.title(f'{model_name} (Seed {seed}) - Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 3, 2) # New subplot for Accuracy
    if 'train_acc' in history and 'val_acc' in history:
        plt.plot(epochs, history['train_acc'], label='Train Accuracy', marker='o', linestyle='-')
        plt.plot(epochs, history['val_acc'], label='Validation Accuracy', marker='o', linestyle='--')
        plt.title(f'{model_name} (Seed {seed}) - Accuracy over Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
    else:
        plt.title(f'{model_name} (Seed {seed}) - Accuracy not available')
        plt.text(0.5, 0.5, 'Accuracy data missing', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

    # Plot AUC
    plt.subplot(1, 3, 3) # Changed to 1 row, 3 columns
    if 'val_auc' in history and all(auc_val is not None for auc_val in history['val_auc']):
        plt.plot(epochs, history['val_auc'], label='Validation AUC', marker='o', color='green', linestyle='-')
        plt.title(f'{model_name} (Seed {seed}) - Validation AUC over Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('AUC')
        plt.legend()
        plt.grid(True)
    else:
        plt.title(f'{model_name} (Seed {seed}) - Validation AUC not available')
        plt.text(0.5, 0.5, 'AUC data missing or invalid', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)

    plt.tight_layout()
    plt.show()

def plot_normalized_confusion_matrix(all_models_results, class_names, model_name):
    if model_name not in all_models_results or not all_models_results[model_name]:
        print(f"No test results found for {model_name} to plot confusion matrix.")
        return

    # Aggregate all true labels and predictions for the given model across all seeds
    all_gts = np.concatenate([np.array(run['gts']) for run in all_models_results[model_name]])
    all_probs = np.concatenate([np.array(run['probs']) for run in all_models_results[model_name]])
    all_preds = np.argmax(all_probs, axis=1)

    cm = confusion_matrix(all_gts, all_preds)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Normalized Confusion Matrix - {model_name} (Aggregated)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


# =========================================================
# 7. CALL NEW PLOTS
# =========================================================
print("\n[STEP 21] Generating Epoch History Plots...")
for model_name in MODEL_ZOO.keys(): # Iterate through defined models
    if model_name in all_histories:
        for seed_val in all_histories[model_name]:
            plot_epoch_history(all_histories, model_name, seed_val)
    else:
        print(f"No history found for {model_name}.")

print("\n[STEP 21] Generating Normalized Confusion Matrices...")
for model_name in MODEL_ZOO.keys(): # Iterate through defined models
    if model_name in all_models_results:
        plot_normalized_confusion_matrix(all_models_results, CLASS_NAMES, model_name)
    else:
        print(f"No test results found for {model_name}.")


# =========================================================
# 8. EXISTING ROC PLOT: PER-CLASS + MICRO ± STD (Moved from 6)
# =========================================================
def plot_combined_roc(all_models_results, class_names):
    for model_name, runs in all_models_results.items():
        if not runs:
            continue
        plt.figure(figsize=(10, 8))
        # ==============================
        # Per-class ROC (mean ± std)
        # ==============================
        for idx, class_name in enumerate(class_names):
            tprs = []
            base_fpr = np.linspace(0,1,101)
            for run in runs:
                y_true = label_binarize(run['gts'], classes=range(len(class_names)))[:, idx]
                y_score = np.array(run['probs'])[:, idx]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                tpr_interp = np.interp(base_fpr, fpr, tpr)
                tpr_interp[0] = 0.0
                tprs.append(tpr_interp)
            mean_tpr = np.mean(tprs, axis=0)
            std_tpr = np.std(tprs, axis=0)
            plt.plot(base_fpr, mean_tpr, label=f'{class_name} (AUC≈{auc(base_fpr, mean_tpr):.4f})', lw=2)
            plt.fill_between(base_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr, alpha=0.2)
        # ==============================
        # Micro-average ROC across all runs
        # ==============================
        all_gts_combined = np.concatenate([np.array(run['gts']) for run in runs])
        all_probs_combined = np.concatenate([np.array(run['probs']) for run in runs])
        y_true_bin = label_binarize(all_gts_combined, classes=range(len(class_names)))
        fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), all_probs_combined.ravel())
        auc_micro = auc(fpr_micro, tpr_micro)
        plt.plot(fpr_micro, tpr_micro, color='black', linestyle='--', label=f'Micro-Average (AUC={auc_micro:.4f})', lw=3)
        # ==============================
        # Chance line & formatting
        # ==============================
        plt.plot([0,1],[0,1],'k:', label='Chance (AUC=0.50)')
        plt.title(f'Combined ROC (Per-Class ± Std + Micro-Average) - {model_name}', fontsize=14)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(loc='lower right', fontsize=10)
        plt.show()

# =========================================================
# 9. EXISTING FNR PLOTTING FUNCTION (Moved from 7)
# =========================================================
def plot_fnr(all_models_results, class_names, title='False Negative Rate per Class'):
    fnr_data_for_plot = []
    for model_name, runs in all_models_results.items():
        if not runs: continue
        all_gts = np.concatenate([np.array(run['gts']) for run in runs])
        all_preds = np.concatenate([np.argmax(np.array(run['probs']), axis=1) for run in runs])
        cm = confusion_matrix(all_gts, all_preds)
        for i, class_name in enumerate(class_names):
            FN = np.sum(cm[i, :]) - cm[i, i]
            TP = cm[i, i]
            fnr = FN / (FN + TP) if (FN + TP) > 0 else 0.0
            fnr_data_for_plot.append({'Model': model_name, 'Class': class_name, 'FNR': fnr})
    df_fnr = pd.DataFrame(fnr_data_for_plot)
    plt.figure(figsize=(12, 7))
    sns.barplot(x='Class', y='FNR', hue='Model', data=df_fnr, palette='viridis')
    plt.title(title)
    plt.ylim(0, df_fnr['FNR'].max() * 1.1 + 0.01)
    plt.xlabel('Class'); plt.ylabel('False Negative Rate (FNR)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(title='Model'); plt.show()

# =========================================================
# 10. CALL EXISTING PLOTS (Moved from 8)
# =========================================================
plot_combined_roc(all_models_results, CLASS_NAMES)
plot_fnr(all_models_results, CLASS_NAMES)

# =========================================================
# 11. SAVE FINAL COMBINED RESULTS (Moved from 9)
# =========================================================
final_results_json = {
    'config': {
        'NUM_CLASSES': NUM_CLASSES,
        'CLASS_FOLDERS': CLASS_FOLDERS,
        'MODEL_ZOO': MODEL_ZOO
    },
    'final_results': all_models_results
}
save_path = os.path.join(RESULTS_DIR, f'final_combined_results.json')
with open(save_path, 'w') as f:
    json.dump(final_results_json, f, indent=4)
print(f"[STEP 21] Final combined results saved to: {save_path} ✅")

print("\n[STEP 21] Comprehensive analysis with JSON plotting completed successfully ✅")
